# Apply Naive Weight-only INT4 Quantization on Qwen3-8b

In [1]:
import tqdm
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from functools import partial
import gc
import os
os.environ['https_proxy'] = 'http://192.168.1.12:7891'

debug = True

if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

/root/workspace/Qwen3.Ink.Cpp/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here we use wikitext-2 dataset for perplexity evaluation. The dataset is automatically downloaded by the code.

In [8]:
print("Loding wikitext datasets ...")
testenc = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', cache_dir="~/.cache/huggingface/datasets")
print("Done")

Loding wikitext datasets ...
Done


In [9]:
def evaluate(model, testenc, tokenizer):
    # we control the text length to avoid error posed by tiktoken
    testenc = tokenizer("\n\n".join(testenc['text']), return_tensors='pt')
    testenc = testenc.input_ids.to(model.device)
    nsamples = 40
    model = model.eval()

    nlls = []
    for i in tqdm.tqdm(range(nsamples), desc="evaluating Qwen on wikitext"):
        batch = testenc[:, (i * 1024):((i + 1) * 1024)].to(model.device)
        with torch.no_grad():
            lm_logits = model(batch).logits
        shift_logits = lm_logits[:, :-1, :].contiguous().float()
        shift_labels = testenc[:, (i * 1024):((i + 1) * 1024)][:, 1:]
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * 1024
        nlls.append(neg_log_likelihood)

    return torch.exp(torch.stack(nlls).sum() / (nsamples * 1024))

# Evaluate the performance of FP32 Qwen
## PPL

In [10]:

model_name = "Qwen/Qwen3-8B"

In [6]:

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)


Loading checkpoint shards: 100%|██████████| 5/5 [00:05<00:00,  1.18s/it]


In [8]:
fp32_perplexity = evaluate(model, testenc, tokenizer)
print(f"\nmodel perplexity: {fp32_perplexity:.2f}")


Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.02it/s]


model perplexity: 10.96


In [9]:
del model
gc.collect()
torch.cuda.empty_cache()

## GSM8k

In [2]:
!lm-eval --tasks gsm8k --model vllm --model_args pretrained=Qwen/Qwen3-8B,max_model_len=8192,dtype=float16 --batch_size auto --trust_remote_code

INFO 06-23 18:17:19 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-23 18:17:19 [__init__.py:239] Automatically detected platform cuda.
2025-06-23:18:17:22 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-06-23:18:17:22 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-06-23:18:17:22 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-06-23:18:17:22 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'Qwen/Qwen3-8B', 'max_model_len': 8192, 'dtype': 'float16', 'trust_remote_code': True}
WARNING 06-23 18:17:23 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 06-23 18:17:28 [config.py:717] This model supports multiple tasks: {'generate', 'score', 'reward', 'classify', 'embed'}. Defaulting to 'generate'.
INFO 06-23 18:17:28 [config.py

# Evaluate Performance Of Int4 Weight-quantized Model
Apply pseudo quantization to check the performance of quantized model directly

In [11]:
def pseudo_quantize_tensor_q4_zero_point(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    zero_point = (-torch.round(min_val / scaling_factor)).clamp_(0, max_int)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor) + zero_point
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q - zero_point) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

def pseudo_quantize_tensor_q40(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_abs_value = torch.amax(w.abs(), dim=-1, keepdim=True)
    max_int = (2 ** (n_bit - 1) - 1)
    min_int = - (2 ** (n_bit - 1) )

    # get the scaling factor, zero point
    scaling_factor = (max_abs_value).clamp(min=1e-5) / (min_int) # (len, 1)
    zero_point = torch.tensor(8).round()
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor)
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo max quantization error: {max_error}")
    return w_f

def pseudo_quantize_tensor_q41(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q = torch.round( 
            (w - min_val) / scaling_factor
        )
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor + min_val
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

quantization_method_dict = {
    "q4z": pseudo_quantize_tensor_q4_zero_point,
    "q40": pseudo_quantize_tensor_q40,
    "q41": pseudo_quantize_tensor_q41
}

@torch.no_grad()
def pseudo_quantize_model_weight(model, w_bit, q_group_size, method):
    q_method = quantization_method_dict[method]
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            # print(f"Pseudo quantizing {n}")
            m.weight.data = q_method(m.weight.data, w_bit, q_group_size)
            # print("")

def quantize_and_evaluate(q_method):

    # load the tokenizer and the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    # Apply fake quantization
    pseudo_quantize_model_weight(model, 4, 32, q_method)
    # Evaluate the model
    model_perplexity = evaluate(model, testenc, tokenizer)
    print(f"\nmodel perplexity: {model_perplexity:.2f}")
    return model, tokenizer


## W41

In [7]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q41")
quantized_model_path = "tmp_quantized_model"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

del model
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.13it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.27it/s]



model perplexity: 12.15


In [13]:
gc.collect()
torch.cuda.empty_cache()

In [14]:
!nvidia-smi

Mon Jun 23 18:40:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
| 30%   38C    P8             29W /  450W |   16094MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [2]:

!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

INFO 06-23 18:29:33 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-23 18:29:33 [__init__.py:239] Automatically detected platform cuda.
2025-06-23:18:29:36 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-06-23:18:29:36 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-06-23:18:29:36 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-06-23:18:29:36 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 06-23 18:29:40 [config.py:717] This model supports multiple tasks: {'score', 'generate', 'classify', 'reward', 'embed'}. Defaulting to 'generate'.
INFO 06-23 18:29:40 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-23 18:29:41 [core.p

In [2]:

!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

INFO 06-23 18:40:45 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-23 18:40:45 [__init__.py:239] Automatically detected platform cuda.
2025-06-23:18:40:48 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-06-23:18:40:48 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-06-23:18:40:48 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-06-23:18:40:48 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 06-23 18:40:53 [config.py:717] This model supports multiple tasks: {'generate', 'embed', 'classify', 'reward', 'score'}. Defaulting to 'generate'.
INFO 06-23 18:40:53 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-23 18:40:54 [core.p

## W4z

In [6]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q4z")
quantized_model_path = "tmp_quantized_model"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

del model
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  2.68it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.26it/s]



model perplexity: 11.49
model size: 4515.90 MiB


In [13]:

!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 05-30 09:30:37 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:09:30:40 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:09:30:40 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:09:30:40 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:09:30:40 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 05-30 09:30:40 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 09:30:40 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 09:30:40 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO

## W40

In [12]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q40")
quantized_model_path = "tmp_quantized_model"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

del model
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.15it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.53it/s]



model perplexity: 11.73


In [2]:

!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

INFO 06-23 19:26:47 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-23 19:26:47 [__init__.py:239] Automatically detected platform cuda.
2025-06-23:19:26:50 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-06-23:19:26:50 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-06-23:19:26:51 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-06-23:19:26:51 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 06-23 19:26:55 [config.py:717] This model supports multiple tasks: {'embed', 'generate', 'reward', 'score', 'classify'}. Defaulting to 'generate'.
INFO 06-23 19:26:55 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-23 19:26:56 [core.p

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    "tmp_quantized_model",
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.92it/s]


remove perviously saved temporary model

In [ ]:
!rm -rf tmp_quantized_model